# 📘 06 — Functions, Lambda, Recursion, Modules
*Part of the Full Stack Data Science & AI booklet. Run cells top to bottom (Shift + Enter). Install libraries first: `pip install -r ../requirements.txt`.*

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import os, sys, math, random, json, time, copy, warnings
from IPython.display import display
warnings.filterwarnings("ignore")

# 7️⃣ Functions
A function is a **reusable block of code**. Write once, use many times.
```python
def name(parameters):
    """docstring"""
    ...
    return value
```
## 7.1 Basics, parameters, return

In [2]:
def greet(name):
    return f"Hello, {name}!"
print(greet("Asha"))

# Default parameter
def power(base, exp=2):
    return base ** exp
print(power(5), power(2, 10), power(exp=3, base=2))   # keyword arguments

# Multiple return values (returns a tuple)
def stats(nums):
    return min(nums), max(nums), sum(nums) / len(nums)
lo, hi, avg = stats([4, 8, 15, 16, 23, 42])
print(lo, hi, avg)

# *args = any number of positional args, **kwargs = any number of keyword args
def add_all(*args):
    return sum(args)
def show(**kwargs):
    for k, v in kwargs.items(): print(k, "=", v)
print(add_all(1, 2, 3, 4))
show(name="Riya", age=20)

# Function without return gives None
def say(): print("hi")
print(say())
help(power)      # shows docstring/signature

Hello, Asha!
25 1024 8
4 42 18.0
10
name = Riya
age = 20
hi
None
Help on function power in module __main__:

power(base, exp=2)
    # Default parameter



#### 🔍 How it works
- `def name(params): ... return value` defines a function; **default parameters** (`exp=2`) make arguments optional; **keyword arguments** (`power(exp=3, base=2)`) allow any order.
- `*args` collects extra positional arguments into a **tuple**; `**kwargs` collects named ones into a **dict**.
- No `return` → returns `None`. Returning several values returns a tuple.
- A **docstring** (`"""..."""`) is the function's manual; `help(f)` displays it.
#### 💼 Where it's used
Functions = reusable, testable code. In data science they wrap cleaning steps, feature engineering and model training; `*args/**kwargs` are everywhere in libraries (e.g. `plt.plot(*args, **kwargs)`).

## 7.2 Scope (LEGB rule)
Python looks for a variable in: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.

In [3]:
x = "global"
def outer():
    x = "enclosing"
    def inner():
        nonlocal x
        x = "changed by inner"
    inner()
    return x
print(outer(), x)

counter = 0
def inc():
    global counter
    counter += 1
inc(); inc(); print(counter)

# Beware: mutable default argument trap
def bad(item, box=[]):
    box.append(item); return box
print(bad(1), bad(2))            # [1] [1, 2]  <- surprise!
def good(item, box=None):
    box = box or []
    box.append(item); return box
print(good(1), good(2))

changed by inner global
2
[1, 2] [1, 2]
[1] [2]


#### 🔍 How it works
- **LEGB**: Python looks for a name in **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in. `global x` lets a function change a global; `nonlocal` changes an enclosing variable.
- **Mutable default trap:** `def bad(item, box=[])` creates the list **once** when the function is defined, so it's shared across calls. Fix: default `None`, then create a new list inside.
#### 💼 Where it's used
This bug appears in real codebases and is a favourite interview question; scope rules explain "why is my variable not changing?".

## 7.3 lambda, map, filter, reduce, sorted key
A **lambda** is a tiny one-line anonymous function.

In [4]:
square = lambda n: n * n
print(square(6))
nums = [1, 2, 3, 4, 5, 6]
print(list(map(lambda n: n * 2, nums)))              # apply to all
print(list(filter(lambda n: n % 2 == 0, nums)))      # keep matching
from functools import reduce
print(reduce(lambda a, b: a * b, nums))              # combine into one
people = [("Riya", 22), ("Arjun", 19), ("Sam", 25)]
print(sorted(people, key=lambda p: p[1]))
print(any(n > 5 for n in nums), all(n > 0 for n in nums))

36
[2, 4, 6, 8, 10, 12]
[2, 4, 6]
720
[('Arjun', 19), ('Riya', 22), ('Sam', 25)]
True True


#### 🔍 How it works
- `lambda n: n*n` = a one-line unnamed function. `map(f, items)` applies f to each; `filter(f, items)` keeps those where f is True; `reduce(f, items)` folds the list into a single value (here the product).
- `sorted(people, key=lambda p: p[1])` sorts by the 2nd field. `any()` = at least one True; `all()` = every one True.
#### 💼 Where it's used
`df.apply(lambda row: ...)`, sorting by custom keys, quick transformations. Modern Python often prefers list comprehensions for the same job.

## 7.4 Recursion — a function that calls itself
Needs a **base case** to stop.

In [5]:
def factorial(n):
    if n <= 1: return 1
    return n * factorial(n - 1)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(factorial(5), [fib(i) for i in range(10)])

# 🔵 Speed up with caching
from functools import lru_cache
@lru_cache(maxsize=None)
def fast_fib(n): return n if n < 2 else fast_fib(n - 1) + fast_fib(n - 2)
print(fast_fib(80))

120 [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
23416728348467685


#### 🔍 How it works
- Recursion = a function calling itself. Needs a **base case** (`n <= 1`) or it never stops.
- `fib(n)` recomputes the same values again and again (exponential time). `@lru_cache` **remembers** results (*memoization*) → `fast_fib(80)` is instant while plain `fib(80)` would take years.
#### 💼 Where it's used
Tree/graph traversal, folder scanning, parsing nested JSON, divide-and-conquer algorithms. Memoization = the idea behind caching in web apps.

## 7.5 🔵 Decorators & generators
- **Decorator**: a function that adds behaviour to another function.
- **Generator** (`yield`): produces values one at a time, saving memory.

In [6]:
import time
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n): return sum(range(n))
print(slow_sum(1_000_000))

def countdown(n):
    while n > 0:
        yield n
        n -= 1
print(list(countdown(5)))
gen = (i * i for i in range(5))      # generator expression
print(next(gen), next(gen))

slow_sum took 0.0093s
499999500000
[5, 4, 3, 2, 1]
0 1


#### 🔍 How it works
- **Decorator:** `@timer` above `def slow_sum` means `slow_sum = timer(slow_sum)`; the `wrapper` runs extra code (timing) before/after the real function without editing it. `*args, **kwargs` let the wrapper accept any arguments.
- **Generator:** `yield` returns one value and **pauses**, resuming on the next request → values are produced *lazily* and never all stored in memory (great for huge files).
#### 💼 Where it's used
Decorators: logging, authentication, caching, timing in Flask/FastAPI/Streamlit (`@st.cache_data`). Generators: reading multi-GB files line by line, data loaders in deep learning.

## 7.6 Modules & packages
A **module** is a `.py` file. A **package** is a folder of modules. Use `import` to reuse code.

In [7]:
import math, random, datetime, os, json
from math import sqrt, pi
import statistics as st

print(math.sqrt(16), math.ceil(2.1), math.floor(2.9), math.factorial(5), math.pi)
print(random.randint(1, 10), random.choice(["a", "b"]), random.sample(range(100), 3))
print(datetime.datetime.now().strftime("%d-%m-%Y %H:%M"))
print(st.mean([1, 2, 3, 4]), st.median([1, 2, 3, 4]))
print(os.getcwd())

# Create your own module
with open("mymath.py", "w") as f:
    f.write("def add(a, b):\n    return a + b\n\nPI = 3.14\n")
import mymath
print(mymath.add(2, 3), mymath.PI)
print(dir(math)[:8])          # explore what a module contains

4.0 3 2 120 3.141592653589793
10 b [6, 58, 28]
07-11-2026 14:17
2.5 2.5
.
5 3.14
['__doc__', '__loader__', '__name__', '__package__', '__spec__', 'acos', 'acosh', 'asin']


#### 🔍 How it works
- `import math` loads a module; `from math import sqrt` imports one name; `import statistics as st` gives a short alias.
- We **created our own module** by writing `mymath.py` and importing it → any `.py` file can be reused. Package = folder of modules; framework = a big package that controls the structure of your app (Django, Streamlit).
- `dir(module)` lists everything inside; `help(module.function)` explains it.
#### 💼 Where it's used
Big projects are split into modules (`cleaning.py`, `models.py`). Knowing the standard library (`datetime`, `json`, `os`, `random`, `collections`) saves you from installing packages.

## 🚀 Mini Project 5: ATM Simulator + Smart Calculator

In [8]:
class_balance = {"balance": 5000}
def atm(action, amount=0):
    if action == "balance":
        return f"Balance: ₹{class_balance['balance']}"
    if action == "deposit":
        class_balance["balance"] += amount
        return f"Deposited ₹{amount}"
    if action == "withdraw":
        if amount > class_balance["balance"]:
            return "❌ Insufficient funds"
        class_balance["balance"] -= amount
        return f"Withdrew ₹{amount}"
    return "Invalid action"

for act, amt in [("balance", 0), ("deposit", 1500), ("withdraw", 7000), ("withdraw", 2000), ("balance", 0)]:
    print(atm(act, amt))

def calc(a, op, b):
    ops = {"+": lambda: a + b, "-": lambda: a - b, "*": lambda: a * b,
           "/": lambda: a / b if b else "Cannot divide by zero"}
    return ops.get(op, lambda: "Bad operator")()
print(calc(10, "+", 5), calc(10, "/", 0), calc(2, "^", 3))

Balance: ₹5000
Deposited ₹1500
❌ Insufficient funds
Withdrew ₹2000
Balance: ₹4500
15 Cannot divide by zero Bad operator


#### 🔍 How it works
- ATM: a **dictionary** stores the balance; one function handles different `action`s with `if`; withdrawing more than the balance is **rejected** (business rule).
- Calculator: a dict maps each operator to a `lambda` (a **dispatch table**) → cleaner than a long `if/elif`; `.get(op, default)` handles unknown operators.
#### 💼 Where it's used
Banking / payment logic and "command routers" in apps and chatbots.

### 🧪 Practice (Section 7)
1. Function to check prime. 2. Function returning list of even numbers. 3. Write `is_palindrome`. 4. Recursive sum of digits. 5. Decorator that prints "Start"/"End".